# 03 — Struggle test (Claim B)
Do teams that have been struggling in a game surge on a late-half drive, beyond what happens at other points? Definitions in `question-doc.md` §6, §7.

## History features
For each drive: what the same team did on its earlier drives in the same game.

In [2]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("../data")
drives = pd.read_parquet(DATA_DIR / "drives.parquet")

drives = drives.sort_values(["game_id", "posteam", "fixed_drive"]).reset_index(drop=True)
g = drives.groupby(["game_id", "posteam"])

drives["prior_n"] = g.cumcount()   # earlier drives by this team in this game
for col in ["is_td", "drive_epa", "first_downs", "three_and_out"]:
    # running total including this drive, minus this drive = earlier drives only
    drives[f"prior_{col}"] = g[col].cumsum().astype(float) - drives[col].astype(float)

drives[["game_id", "posteam", "fixed_drive", "prior_n", "prior_is_td", "prior_drive_epa"]].head(8)

,game_id,posteam,fixed_drive,prior_n,prior_is_td,prior_drive_epa
0,2016_01_BUF_BAL,BAL,1.0,0,0.0,0.000000
1,2016_01_BUF_BAL,BAL,3.0,1,0.0,-1.424516
2,2016_01_BUF_BAL,BAL,5.0,2,0.0,-5.283258
3,2016_01_BUF_BAL,BAL,7.0,3,0.0,-3.314834
4,2016_01_BUF_BAL,BAL,9.0,4,1.0,2.161842
5,2016_01_BUF_BAL,BAL,11.0,5,1.0,1.513576
6,2016_01_BUF_BAL,BAL,13.0,6,1.0,-1.020504
7,2016_01_BUF_BAL,BAL,15.0,7,1.0,-2.583396


## Struggled flags (§6)
Both definitions. Minimum 3 earlier drives in the same game; league averages are per season.

In [3]:
season_avg = drives.groupby("season").agg(avg_epa=("drive_epa", "mean"), avg_fd=("first_downs", "mean"))
drives = drives.merge(season_avg, on="season", how="left")

enough = drives["prior_n"] >= 3

# Definition 1 (scoring): no TDs, and below-average EPA per drive
drives["struggled_1"] = (enough & (drives["prior_is_td"] == 0)
                         & (drives["prior_drive_epa"] / drives["prior_n"] < drives["avg_epa"]))

# Definition 2 (ball movement): below-average first downs per drive, OR at least half three-and-outs
drives["struggled_2"] = (enough & ((drives["prior_first_downs"] / drives["prior_n"] < drives["avg_fd"])
                                   | (drives["prior_three_and_out"] / drives["prior_n"] >= 0.5)))

print("share with 3+ earlier drives:", round(enough.mean(), 3))
drives[["struggled_1", "struggled_2"]].mean().round(3)

share with 3+ earlier drives: 0.717


struggled_1    0.166
struggled_2    0.398
dtype: float64

In [4]:
e = drives[enough].copy()
fd_rate = e["prior_first_downs"] / e["prior_n"]
q25 = fd_rate.groupby(e["season"]).transform(lambda s: s.quantile(0.25))

pd.Series({
    "def2: below-avg first downs":       (fd_rate < e["avg_fd"]).mean(),
    "def2: >= half three-and-outs":      (e["prior_three_and_out"] / e["prior_n"] >= 0.5).mean(),
    "alt: bottom-quartile first downs":  (fd_rate < q25).mean(),
    "def1: no TD + below-avg EPA":       e["struggled_1"].mean(),
}).round(3)

def2: below-avg first downs         0.548
def2: >= half three-and-outs        0.112
alt: bottom-quartile first downs    0.247
def1: no TD + below-avg EPA         0.232
dtype: float64

In [5]:
# §6 Def 2 (v1.3): bottom-quartile first downs/drive for the season, OR >= half earlier drives three-and-out
n = drives["prior_n"].clip(lower=1)  # avoid divide-by-zero on a team's first drive
tao = drives["prior_three_and_out"] / n

drives["fd_rate"] = (drives["prior_first_downs"] / n).where(enough)
q25 = drives.loc[enough].groupby("season")["fd_rate"].quantile(0.25)  # cutoff per season, among drives with 3+ earlier drives
drives["fd_q25"] = drives["season"].map(q25)

drives["struggled_2_loose"] = enough & ((drives["fd_rate"] < drives["avg_fd"]) | (tao >= 0.5))  # original version
drives["struggled_2"] = enough & ((drives["fd_rate"] <= drives["fd_q25"]) | (tao >= 0.5))       # v1.3

sub = drives[enough]
print(sub[["struggled_1", "struggled_2", "struggled_2_loose"]].mean().round(3))
print("overlap, share of Def 2 that is also Def 1:", round((sub["struggled_1"] & sub["struggled_2"]).sum() / sub["struggled_2"].sum(), 3))

struggled_1          0.232
struggled_2          0.287
struggled_2_loose    0.555
dtype: float64
overlap, share of Def 2 that is also Def 1: 0.514


In [6]:
# §6 timing label: late (<= 240 s left at start), opener (team's first 2nd-half drive), mid (everything else)
half_n = drives.groupby(["game_id", "posteam", "game_half"]).cumcount()
drives["label"] = "mid"
drives.loc[(drives["game_half"] == "Half2") & (half_n == 0), "label"] = "opener"
drives.loc[drives["start_secs"] <= 240, "label"] = "late"  # late wins if both apply

struggled_any = enough & drives["eligible"] & (drives["struggled_1"] | drives["struggled_2"])
print(pd.crosstab(drives.loc[struggled_any, "game_half"], drives.loc[struggled_any, "label"]))

label      late   mid  opener
game_half                    
Half1      1954  4208       0
Half2       802  5558    1964


In [8]:
import numpy as np

# §7: 1st half, drives after a struggle, late vs mid. Selected on history only (struggled_*), never on this drive's outcome.
h1 = drives[(drives["game_half"] == "Half1") & enough & drives["eligible"] & drives["label"].isin(["late", "mid"])].copy()
h1["any_score"] = h1["result"].isin(["Touchdown", "Field goal"])
metrics = ["is_td", "any_score", "drive_epa"]
h1[["is_td", "any_score"]] = h1[["is_td", "any_score"]].astype(float)  # so they can be averaged

def late_vs_mid(df, B=2000, seed=0):
    # per-game sums and drive counts, so each bootstrap draw is just a re-sum of games
    sums = df.groupby(["game_id", "label"])[metrics].sum().unstack("label", fill_value=0)
    cnt = df.groupby(["game_id", "label"]).size().unstack("label", fill_value=0)
    s_late, s_mid = (sums.xs(l, axis=1, level="label").to_numpy() for l in ["late", "mid"])
    n_late, n_mid = cnt["late"].to_numpy(), cnt["mid"].to_numpy()

    late = s_late.sum(0) / n_late.sum()
    mid = s_mid.sum(0) / n_mid.sum()
    idx = np.random.default_rng(seed).integers(0, len(cnt), size=(B, len(cnt)))  # B resamples of games
    boot = s_late[idx].sum(1) / n_late[idx].sum(1)[:, None] - s_mid[idx].sum(1) / n_mid[idx].sum(1)[:, None]
    lo, hi = np.percentile(boot, [2.5, 97.5], axis=0)
    return pd.DataFrame({"late": late, "mid": mid, "gap": late - mid, "lo": lo, "hi": hi}, index=metrics).round(3)

for d in ["struggled_1", "struggled_2", "struggled_2_loose"]:
    print(d, "| late drives:", int((h1[d] & (h1["label"] == "late")).sum()), "| mid drives:", int((h1[d] & (h1["label"] == "mid")).sum()))
    print(late_vs_mid(h1[h1[d]]), "\n")

struggled_1 | late drives: 1335 | mid drives: 3164
            late    mid    gap     lo     hi
is_td      0.144  0.225 -0.081 -0.104 -0.058
any_score  0.349  0.381 -0.032 -0.062 -0.003
drive_epa  0.161 -0.121  0.282  0.100  0.454 

struggled_2 | late drives: 1451 | mid drives: 3298
            late    mid    gap     lo     hi
is_td      0.136  0.229 -0.092 -0.115 -0.070
any_score  0.340  0.385 -0.045 -0.076 -0.018
drive_epa  0.097 -0.082  0.179  0.000  0.348 

struggled_2_loose | late drives: 2372 | mid drives: 4814
            late    mid    gap     lo     hi
is_td      0.148  0.234 -0.086 -0.104 -0.067
any_score  0.353  0.390 -0.037 -0.059 -0.014
drive_epa  0.188 -0.042  0.230  0.093  0.374 



In [9]:
# control: same late-vs-mid table for drives NOT flagged (still 3+ earlier drives), per definition
for d in ["struggled_1", "struggled_2", "struggled_2_loose"]:
    ctrl = h1[~h1[d]]
    print(d, "NOT flagged | late drives:", int((ctrl["label"] == "late").sum()), "| mid drives:", int((ctrl["label"] == "mid").sum()))
    print(late_vs_mid(ctrl), "\n")

struggled_1 NOT flagged | late drives: 2936 | mid drives: 4096
            late    mid    gap     lo     hi
is_td      0.170  0.258 -0.088 -0.106 -0.068
any_score  0.386  0.422 -0.037 -0.057 -0.014
drive_epa  0.375  0.173  0.202  0.072  0.340 

struggled_2 NOT flagged | late drives: 2820 | mid drives: 3962
            late    mid    gap     lo     hi
is_td      0.174  0.255 -0.081 -0.101 -0.062
any_score  0.392  0.420 -0.028 -0.052 -0.006
drive_epa  0.417  0.150  0.266  0.117  0.406 

struggled_2_loose NOT flagged | late drives: 1899 | mid drives: 2446
            late    mid    gap     lo     hi
is_td      0.179  0.262 -0.083 -0.108 -0.058
any_score  0.401  0.432 -0.031 -0.060 -0.002
drive_epa  0.458  0.216  0.242  0.053  0.416 



In [10]:
import numpy as np
import pandas as pd

METRICS = ["is_td", "any_score", "drive_epa"]
FLAGS = ["struggled_1", "struggled_2", "struggled_2_loose"]

def add_history(drives, lookback=None):
    # history from EARLIER drives in the same game only; lookback=k uses just the last k drives
    d = drives.sort_values(["game_id", "posteam", "fixed_drive"]).reset_index(drop=True)
    cols = ["is_td", "drive_epa", "first_downs", "three_and_out"]
    d["_one"] = 1.0
    for c in cols:
        d[f"_{c}"] = d[c].astype(float)
    g = d.groupby(["game_id", "posteam"])
    for c in ["one"] + cols:
        col = f"_{c}"
        if lookback is None:
            d[f"p_{c}"] = g[col].cumsum() - d[col]
        else:
            d[f"p_{c}"] = g[col].transform(lambda s: s.shift().rolling(lookback, min_periods=1).sum()).fillna(0)
    n = d["p_one"].clip(lower=1)
    d["enough"] = d["p_one"] >= 3
    enough = d["enough"]
    avg_epa = d.groupby("season")["drive_epa"].transform("mean")
    avg_fd = d.groupby("season")["first_downs"].transform("mean")
    fd = (d["p_first_downs"] / n).where(enough)
    q25 = fd[enough].groupby(d.loc[enough, "season"]).quantile(0.25)
    tao = d["p_three_and_out"] / n >= 0.5
    d["struggled_1"] = enough & (d["p_is_td"] == 0) & (d["p_drive_epa"] / n < avg_epa)
    d["struggled_2"] = enough & ((fd <= d["season"].map(q25)) | tao)
    d["struggled_2_loose"] = enough & ((fd < avg_fd) | tao)
    return d

def add_labels(d):
    # late (<= 240 s left), opener (team's first 2nd-half drive), mid (everything else)
    half_n = d.groupby(["game_id", "posteam", "game_half"]).cumcount()
    d["label"] = "mid"
    d.loc[(d["game_half"] == "Half2") & (half_n == 0), "label"] = "opener"
    d.loc[d["start_secs"] <= 240, "label"] = "late"
    return d

def prep(d):
    d = d.copy()
    d["any_score"] = d["result"].isin(["Touchdown", "Field goal"])
    d[["is_td", "any_score"]] = d[["is_td", "any_score"]].astype(float)
    return d

def did(df, flag, half, a="late", b="mid", metrics=METRICS, B=1000, seed=0):
    # gap (a minus b) among flagged drives, same gap among unflagged, and their difference; 95% interval resampling games
    x = df[(df["game_half"] == half) & df["enough"] & df["label"].isin([a, b])]
    games = pd.Index(x["game_id"].unique())
    G = len(games)
    cells = {}
    for grp, m in [("s", x[flag]), ("n", ~x[flag])]:
        for lab in [a, b]:
            sub = x[m & (x["label"] == lab)]
            sums = sub.groupby("game_id")[metrics].sum().reindex(games, fill_value=0).to_numpy()
            cnt = sub.groupby("game_id").size().reindex(games, fill_value=0).to_numpy().astype(float)
            cells[(grp, lab)] = (sums, cnt)
    def gaps(idx):
        out = []
        for grp in ["s", "n"]:
            mean = {lab: cells[(grp, lab)][0][idx].sum(-2) / np.maximum(cells[(grp, lab)][1][idx].sum(-1), 1)[..., None]
                    for lab in [a, b]}
            out.append(mean[a] - mean[b])
        return out
    gs, gn = gaps(np.arange(G))
    idx = np.random.default_rng(seed).integers(0, G, size=(B, G))
    bs, bn = gaps(idx)
    lo, hi = np.percentile(bs - bn, [2.5, 97.5], axis=0)
    return pd.DataFrame({"gap_struggled": gs, "gap_not": gn, "diff": gs - gn, "lo": lo, "hi": hi}, index=metrics).round(3)

d_all = add_history(drives)
print("sanity, should match 0.232 / 0.287 / 0.555:")
print(d_all[d_all["enough"]][FLAGS].mean().round(3))

d0 = prep(add_labels(d_all))
d0 = d0[d0["eligible"]]
blocks = {}
for half, comps in [("Half1", [("late", "mid")]), ("Half2", [("late", "mid"), ("opener", "mid")])]:
    for flag in FLAGS:
        for a, b in comps:
            blocks[(half, flag, f"{a}-{b}")] = did(d0, flag, half, a, b)
print(pd.concat(blocks, names=["half", "definition", "comparison"]).to_string())

sanity, should match 0.232 / 0.287 / 0.555:
struggled_1          0.232
struggled_2          0.287
struggled_2_loose    0.555
dtype: float64
                                              gap_struggled  gap_not   diff     lo     hi
half  definition        comparison                                                       
Half1 struggled_1       late-mid   is_td             -0.081   -0.088  0.007 -0.025  0.036
                                   any_score         -0.032   -0.037  0.005 -0.034  0.046
                                   drive_epa          0.282    0.202  0.080 -0.146  0.309
      struggled_2       late-mid   is_td             -0.092   -0.081 -0.012 -0.042  0.018
                                   any_score         -0.045   -0.028 -0.017 -0.056  0.021
                                   drive_epa          0.179    0.266 -0.087 -0.322  0.141
      struggled_2_loose late-mid   is_td             -0.086   -0.083 -0.003 -0.035  0.030
                                   any_score      

In [11]:
plays = pd.read_parquet("../data/plays.parquet", columns=["game_id", "fixed_drive", "posteam", "game_half", "epa", "is_late"])
keep = d_all[["game_id", "fixed_drive", "posteam", "enough"] + FLAGS]
p = plays.merge(keep, on=["game_id", "fixed_drive", "posteam"], how="inner")  # drops plays in drives we removed
p["label"] = np.where(p["is_late"], "late", "rest")
for half in ["Half1", "Half2"]:
    for flag in FLAGS:
        print(half, flag)
        print(did(p, flag, half, "late", "rest", metrics=["epa"]).to_string())

Half1 struggled_1
     gap_struggled  gap_not   diff     lo     hi
epa          0.067    0.074 -0.007 -0.042  0.026
Half1 struggled_2
     gap_struggled  gap_not   diff     lo     hi
epa          0.059    0.079 -0.019 -0.052  0.017
Half1 struggled_2_loose
     gap_struggled  gap_not   diff     lo     hi
epa          0.062    0.087 -0.025 -0.058  0.011
Half2 struggled_1
     gap_struggled  gap_not   diff     lo     hi
epa         -0.021   -0.025  0.004 -0.046  0.054
Half2 struggled_2
     gap_struggled  gap_not   diff     lo     hi
epa         -0.004   -0.027  0.023 -0.016  0.062
Half2 struggled_2_loose
     gap_struggled  gap_not   diff     lo     hi
epa         -0.009   -0.032  0.023 -0.007  0.052


In [12]:
def variant(lookback=None, min_secs=30, drop_cutoff=False, team_demean=False, n_range=None):
    d = add_labels(add_history(drives, lookback))
    d = d[d["start_secs"] >= min_secs]
    if drop_cutoff:
        d = d[d["result"] != "End of half"]
    if n_range:
        d = d[d["p_one"].between(*n_range)]
    d = prep(d)
    if team_demean:  # subtract each team-season's own average, so teams are compared with themselves
        for m in METRICS:
            d[m] = d[m] - d.groupby(["season", "posteam"])[m].transform("mean")
    return d

variants = {
    "base": {},
    "lookback 4": dict(lookback=4),
    "lookback 6": dict(lookback=6),
    "lookback 8": dict(lookback=8),
    "min 20 s": dict(min_secs=20),
    "min 25 s": dict(min_secs=25),
    "no cut-off drives": dict(drop_cutoff=True),
    "team vs itself": dict(team_demean=True),
    "3-4 prior drives": dict(n_range=(3, 4)),
    "5+ prior drives": dict(n_range=(5, 99)),
}
rows = {}
for name, kw in variants.items():
    d = variant(**kw)
    for flag in FLAGS:
        t = did(d, flag, "Half1", B=500)
        rows[(name, flag)] = {m: f"{t.loc[m, 'diff']:+.3f} [{t.loc[m, 'lo']:+.3f}, {t.loc[m, 'hi']:+.3f}]" for m in METRICS}
print(pd.DataFrame(rows).T.to_string())

                                                       is_td                any_score                drive_epa
base              struggled_1        +0.007 [-0.023, +0.036]  +0.005 [-0.030, +0.041]  +0.080 [-0.142, +0.310]
                  struggled_2        -0.012 [-0.043, +0.018]  -0.017 [-0.056, +0.019]  -0.087 [-0.328, +0.139]
                  struggled_2_loose  -0.003 [-0.035, +0.031]  -0.006 [-0.042, +0.035]  -0.011 [-0.247, +0.228]
lookback 4        struggled_1        +0.003 [-0.026, +0.034]  +0.002 [-0.031, +0.038]  +0.037 [-0.187, +0.277]
                  struggled_2        -0.006 [-0.039, +0.023]  -0.018 [-0.055, +0.022]  -0.074 [-0.332, +0.155]
                  struggled_2_loose  -0.012 [-0.042, +0.020]  -0.020 [-0.057, +0.017]  -0.077 [-0.311, +0.137]
lookback 6        struggled_1        +0.005 [-0.024, +0.034]  +0.004 [-0.030, +0.042]  +0.069 [-0.150, +0.290]
                  struggled_2        -0.014 [-0.047, +0.015]  -0.016 [-0.053, +0.020]  -0.085 [-0.349, +0.137]
 

In [ ]:


















      .  














xdwcrcv
v




